In [ ]:
import pandas as pd

# =====================================================
# 1. 기존 BERTopic 결과 불러오기
# =====================================================

INPUT_FILE = "liverpool_cafe_bertopic_result.xlsx"
OUTPUT_FILE = "liverpool_cafe_bertopic_macro_result.xlsx"

# 저장할 때 sheet_name을 다르게 했을 수 있으니 우선 시트 확인
xls = pd.ExcelFile(INPUT_FILE)
print(xls.sheet_names)

# 보통 topic_labeled_data 또는 cafe_topic_labeled_data에 원본+topic 결과가 있음
SHEET_NAME = "cafe_topic_labeled_data" if "cafe_topic_labeled_data" in xls.sheet_names else xls.sheet_names[0]

topic_df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(topic_df.shape)
print(topic_df.columns.tolist())

display(topic_df.head())

In [ ]:
# =====================================================
# 2. topic_name 없으면 생성
# =====================================================

if "topic_name" not in topic_df.columns:
    if "Name" in topic_df.columns:
        topic_df["topic_name"] = topic_df["Name"]
    else:
        topic_df["topic_name"] = topic_df["topic"].astype(str)

print(topic_df[["topic", "topic_name"]].drop_duplicates().head(30))

In [ ]:
# =====================================================
# 세부 토픽 -> 상위 토픽 매핑: 키워드 점수 기반
# 리버풀 카페 BERTopic 결과 기준
# =====================================================

macro_keywords = {
    "감독/전술/운영 방향": [
        "슬롯", "전술", "경질", "유임", "감독", "선임", "알론소", "이라올라",
        "엔리케", "로저스", "휴즈", "헤드", "매니저", "코치", "단장",
        "권한", "부임", "개선", "데이터", "인터뷰", "라인업", "기용"
    ],

    "이적시장/선수보강": [
        "이적", "영입", "계약", "가격", "주급", "매물", "판매", "방출",
        "보강", "대체", "잔류", "자계", "옵션", "구단", "시장",
        "프림퐁", "케르케즈", "비르츠", "이삭", "엔소", "엔조", "앤더슨",
        "코나테", "자케", "반헤케", "레오니", "세네시", "디오망데",
        "바르콜라", "고든", "보웬", "올리세", "흐비차", "와튼",
        "에키", "키에사", "각포", "마페", "스콧", "바르콜라"
    ],

    "선수 평가/스쿼드": [
        "알리송", "알리송베케르",
    "조고메즈", "고메즈",
    "엔도", "엔도와타루",
    "반다이크", "버질반다이크",
    "코나테",
    "케르케즈",
    "비르츠", "플로리안비르츠",
    "소보슬라이", "소보슬러이",
    "이삭", "이사크",
    "맥알리스터", "맥알",
    "살라", "모하메드살라",
    "브래들리",
    "키에사",
    "레오니",
    "커티스존스", "커존",
    "각포", "학포", "가크포",
    "에키티케",
    "마마르다슈빌리", "마마슈",
    "로버트슨",
    "우드먼",
    "프림퐁",
    "그라벤베르흐", "흐라벤베르흐",
    "트레이뇨니",
    "리오은구모하"
    ],

    "경기력/전술 분석": [
        "패스", "중원", "수비", "공격", "압박", "빌드업", "경합",
        "세트", "피스", "역할", "장면", "체계", "측면", "교체",
        "스쿼드", "포메이션", "전방", "흐름", "경기"
    ],

    "시즌 성적/대회 결과": [
        "우승", "챔스", "진출", "승점", "순위", "유로파", "강등",
        "예선", "득실", "맞대결", "세리에", "밀란", "유베",
        "본머스", "안필드", "강팀", "더블", "경쟁", "확정",
        "마지막", "결과", "기록"
    ],

    "레전드/클럽 역사": [
        "클롭", "제라드", "캐러거", "아게르", "베니테즈", "케니",
        "이스탄불", "레전드", "고별", "시대", "시절", "추억",
        "입문", "아이콘", "역사", "ynwa", "안필드"
    ],

    "부상/복귀/컨디션": [
        "부상", "복귀", "햄스트링", "출전", "위험", "불운",
        "기량", "컨디션", "전성기", "나이", "세월", "교체"
    ],

    "팬 반응/커뮤니티": [
        "감사", "의견", "공감", "동의", "동감", "마음", "행복",
        "환영", "인사", "고생", "리스펙", "투표", "궁금증",
        "정보", "말씀", "인정", "찬성"
    ],

    "구단 논란/민심": [
        "최악", "논란", "불만", "짜증", "후회", "실수", "실패",
        "민심", "이별", "작별", "패싱", "해외", "취소",
        "싸인", "해지", "구독", "반대", "아웃", "차별",
        "혐오", "비하", "부정", "카페", "반복"
    ],

    "유니폼/굿즈/팬 활동": [
        "유니폼", "어센틱", "마킹", "홈그", "액자", "사진",
        "구매", "부탁", "방법", "듀오", "fa", "결승"
    ]
}


def assign_macro_topic_score(topic_name):
    t = str(topic_name).lower()

    if "기타" in t or "-1" in t:
        return "기타/분류불가"

    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k.lower() for k in keywords if k.lower() in t]
        scores[macro] = len(matched)

    best_macro = max(scores, key=scores.get)
    best_score = scores[best_macro]

    if best_score == 0:
        return "기타"

    return best_macro

In [ ]:
# # =====================================================
# # 세부 토픽 -> 상위 토픽 매핑: 키워드 점수 기반
# # =====================================================

# macro_keywords = {
#     "이적시장/선수보강": [
#         "이적", "시장", "영입", "임대", "계약", "오피셜", "주급", "가격",
#         "이적료", "디렉터", "레비", "랑게", "비나이", "오시멘", "레앙",
#         "은디아예", "스콧", "기라시", "쿠두스", "오도베르", "사비뉴",
#         "요렌테", "팔리냐", "갤러거", "무아니", "래시포드", "유망주",
#         "매물", "방출", "판매", "보강", "스트라이커", "공미", "공격수",
#         "윙어", "티아고", "요케레스", "누사", "히샬리송"
#     ],

#     "잔류/강등/시즌 결과": [
#         "잔류", "강등", "승점", "순위", "마지막", "확정", "원정",
#         "더블", "coys", "고생", "마음", "승리", "대항전", "기록",
#         "패배", "우승", "경쟁", "중위"
#     ],

#     "감독/전술 방향성": [
#         "데제르비", "투도르", "프랭크", "전술", "압박", "연계",
#         "빌드업", "라인업", "교체", "성향", "선임", "감독"
#     ],

#     "선수 기량/포지션 평가": [
#         # 플레이/포지션/능력 관련 일반 키워드
#         "플레이", "능력", "왼발", "오른발", "속도", "피지컬", "민첩성",
#         "헤더", "공격", "수비", "가담", "압박", "패스", "득점력",
#         "포지션", "미드필더", "공미", "윙어", "스트라이커", "공격수",
#         "풀백", "센터백", "키퍼", "골키퍼", "주전", "백업",
#         "프로", "비판", "태도", "실수", "단점", "장점",

#         # 기존에 있던 핵심 선수/표현
#         "손흥민", "시몬스", "사비시몬스", "그레이", "텔",
#         "히샬리송", "히샬", "히살리송",
#         "로메로", "포로", "판더펜", "판더벤", "반더벤",
#         "베리발", "비수마", "벤탕쿠르",

#         # 감정분석 프롬프트에 있던 토트넘 선수단
#         "비카리오", "굴리엘모", "드라구신", "라구드라구신",
#         "단소", "케빈단소",
#         "제임스매디슨", "매디슨",
#         "마티스텔", "데스티니우도기", "우도기",
#         "아치그레이", "루카스베리발",
#         "크리스티안로메로", "도미닉솔랑케", "솔랑케",
#         "모하메드쿠드스", "쿠드스", "쿠두스",
#         "데얀클루셉스키", "클루셉스키", "클루셉", "셉셉",
#         "코너갤러거", "갤러거",
#         "페드로포로", "제드스펜스", "스펜스",
#         "윌손오도베르", "오도베르",
#         "파페마타르사르", "사르",
#         "로드리고벤탕쿠르",
#         "안토닌킨스키", "킨스키",
#         "벤데이비스", "데이비스",
#         "미키판더펜",
#         "소우자",
#         "랑달콜로무아니", "콜로무아니", "무아니",
#         "브랜던오스틴", "오스틴"
#     ],

#     "구단 운영/보드진 비판": [
#         "보드진", "수뇌부", "구단주", "무능", "운영",
#         "실망", "최악", "위험", "충격", "레비", "랑게", "비나이"
#     ],

#     "부상/복귀 이슈": [
#         "부상", "회복", "복귀", "의료진", "메디컬", "클루셉스키",
#         "장기", "관리"
#     ],

#     "골키퍼/수비진 평가": [
#         "킨스키", "키퍼", "골키퍼", "선방", "주전", "비카리오",
#         "페르브뤼헌", "트래포드", "달로우", "수비진"
#     ],

#     "팬 반응/소통": [
#         "감사", "덕분", "의견", "설명", "공감", "동의",
#         "동감", "행복", "나이스", "수고"
#     ]
# }


# def assign_macro_topic_score(topic_name):
#     t = str(topic_name).lower()

#     if "기타" in t or "-1" in t:
#         return "기타/분류불가"

#     scores = {}

#     for macro, keywords in macro_keywords.items():
#         matched = [k for k in keywords if k in t]
#         scores[macro] = len(matched)

#     # 가장 많이 매칭된 카테고리 선택
#     best_macro = max(scores, key=scores.get)
#     best_score = scores[best_macro]

#     # 아무 키워드도 안 걸리면 기타
#     if best_score == 0:
#         return "기타"

#     return best_macro

In [ ]:
topic_df["macro_topic"] = topic_df["topic_name"].apply(assign_macro_topic_score)

display(
    topic_df[["topic", "topic_name", "macro_topic"]]
    .drop_duplicates()
    .sort_values("topic")
)

In [ ]:
def get_macro_scores(topic_name):
    t = str(topic_name).lower()

    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = {
            "score": len(matched),
            "matched_keywords": matched
        }

    return scores


# topic별 점수 확인용 테이블 만들기
score_rows = []

for _, row in (
    topic_df[["topic", "topic_name"]]
    .drop_duplicates()
    .sort_values("topic")
    .iterrows()
):
    scores = get_macro_scores(row["topic_name"])

    score_rows.append({
        "topic": row["topic"],
        "topic_name": row["topic_name"],
        "macro_topic": assign_macro_topic_score(row["topic_name"]),
        "scores": scores
    })

score_df = pd.DataFrame(score_rows)

display(score_df)

In [ ]:
# # =====================================================
# # 5. 필요하면 topic 번호 기준 수동 보정
# # =====================================================

# manual_override = {
#     # 예시: topic 번호: "상위 토픽명"
#     # 7: "선수 기량/포지션 평가",
#     # 8: "이적시장/선수보강",
#     # 10: "이적시장/선수보강",
#     # 11: "이적시장/선수보강",
#     # 18: "이적시장/선수보강",
#     # 22: "이적시장/선수보강",
# }

# for topic_id, macro in manual_override.items():
#     topic_df.loc[topic_df["topic"] == topic_id, "macro_topic"] = macro

# display(
#     topic_df[["topic", "topic_name", "macro_topic"]]
#     .drop_duplicates()
#     .sort_values("topic")
# )

In [ ]:
# =====================================================
# 6. 38R 전후 상위 토픽 비교
# =====================================================

period_count = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"]
)

period_ratio = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"],
    normalize="index"
) * 100

display(period_count)
display(period_ratio.round(2))

In [ ]:
# =====================================================
# 7. 긍정/부정별 상위 토픽 비교
# =====================================================

pn_df = topic_df[topic_df["gpt_sentiment"].isin(["positive", "negative"])].copy()

sentiment_count = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"]
)

sentiment_ratio = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"],
    normalize="index"
) * 100

display(sentiment_count)
display(sentiment_ratio.round(2))

In [ ]:
# =====================================================
# 8. 저장
# =====================================================

topic_macro_check = (
    topic_df
    .groupby(["topic", "topic_name", "macro_topic"])
    .size()
    .reset_index(name="count")
    .sort_values("topic")
)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    topic_df.to_excel(writer, sheet_name="topic_macro_data", index=False)
    topic_macro_check.to_excel(writer, sheet_name="topic_macro_check", index=False)
    period_count.to_excel(writer, sheet_name="period_macro_count")
    period_ratio.round(2).to_excel(writer, sheet_name="period_macro_ratio")
    sentiment_count.to_excel(writer, sheet_name="sentiment_macro_count")
    sentiment_ratio.round(2).to_excel(writer, sheet_name="sentiment_macro_ratio")

print("저장 완료:", OUTPUT_FILE)